# TabPFN Regression — DIMER artifact inference tutorial

[GitHub](https://github.com/kurtvalcorza/tabpfn-regressor-pipeline) · [Open in Colab](https://colab.research.google.com/github/kurtvalcorza/tabpfn-regressor-pipeline/blob/main/tutorials/tabpfn_regressor_artifact_inference_colab.ipynb) · [Model card](https://github.com/kurtvalcorza/tabpfn-regressor-pipeline/blob/main/MODEL_CARD.md)

**Profile:** `ARTIFACT-INFERENCE` · **DIMER Notebook Specification:** 1.0

This notebook consumes a DIMER artifact bundle — `artifact_manifest.json`, `model.tabpfn_fit`, `model.ckpt` — produced **outside this execution** by the TabPFN fine-tuner worker, validates it before any model state is deserialized, reconstructs the estimator through the repository's serving loader, accepts genuinely new unlabelled data, predicts, and exports results. No training, fine-tuning, or in-context refitting occurs: the fitted archive already contains the training rows the estimator conditions on, and the loader restores that state as-is.

**Trust boundary.** Manifest digest checks establish that the three files are internally consistent, not that the sender is trustworthy. `model.tabpfn_fit` is a ZIP of JSON parameters plus serialized Python/torch estimator state and `model.ckpt` is a torch checkpoint; loading them executes trusted model state, and path-safety checks on the archive do not change that. Load only artifacts from a producer you trust, and supply the whole-bundle digests you were given.

**Prerequisites:** the three artifact files from a separate producing run; a separate CSV of new rows with exactly the artifact's feature columns; a clean Python 3.11+ runtime with the tutorial lock set; CPU is sufficient. Uploaded data stays in the runtime; do not upload restricted data to an unauthorized environment. Regression only; no classification and no per-prediction uncertainty — `predict()` returns a raw point estimate (negative values are valid), with no interval.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys
if sys.version_info < (3, 11):
    raise RuntimeError("This tutorial requires Python 3.11+.")
PIPE_DIR = Path("/content/tabpfn-regressor-pipeline")
if PIPE_DIR.exists():
    shutil.rmtree(PIPE_DIR)
!git clone -q https://github.com/kurtvalcorza/tabpfn-regressor-pipeline.git /content/tabpfn-regressor-pipeline
%pip install -q -r /content/tabpfn-regressor-pipeline/tutorials/requirements-colab.txt
!git -C /content/tabpfn-regressor-pipeline rev-parse HEAD

## 1. Runtime, external artifact upload, and pre-load validation

Upload exactly `artifact_manifest.json`, `model.tabpfn_fit`, and `model.ckpt`. Before anything is deserialized the notebook checks the manifest schema and task type, that both binary files match the manifest's SHA-256 digests and that their sizes are plausible, that the fitted archive is a ZIP whose members are all safe relative paths and which carries `init_params.json`, and it prints the feature schema the artifact expects. Optionally paste the digests you were given out-of-band into `EXPECTED_*_SHA256`; a mismatch fails closed.

In [ ]:
import hashlib, platform, zipfile, importlib.metadata as md
from pathlib import PurePosixPath
import numpy as np, pandas as pd, torch
from google.colab import files
sys.path.insert(0, str(PIPE_DIR))
from serving.load_artifact import load_dimer_tabpfn_artifact, rewrite_model_path
IS_CLASSIFIER = False
EXPECTED_FITTED_SHA256 = ""  # @param {type:"string"}
EXPECTED_CKPT_SHA256 = ""  # @param {type:"string"}
print("Python", platform.python_version(), "torch", md.version("torch"), "tabpfn", md.version("tabpfn"), "pandas", md.version("pandas"))
print("Device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
ART = Path("/content/external-tabpfn-artifact")
if ART.exists():
    shutil.rmtree(ART)
ART.mkdir(parents=True)
uploaded = files.upload()
required = {"artifact_manifest.json", "model.tabpfn_fit", "model.ckpt"}
if set(uploaded) != required:
    raise ValueError(f"Upload exactly {sorted(required)}; got {sorted(uploaded)}")
for name, payload in uploaded.items():
    (ART / name).write_bytes(payload)
manifest = json.loads((ART / "artifact_manifest.json").read_text(encoding="utf-8"))
if manifest.get("schemaVersion") != 1 or manifest.get("taskType") != "tabular_regression":
    raise ValueError(f"unsupported manifest: schemaVersion={manifest.get('schemaVersion')} taskType={manifest.get('taskType')}")
for key, expected_override in (("fittedEstimator", EXPECTED_FITTED_SHA256), ("foundationCheckpoint", EXPECTED_CKPT_SHA256)):
    path = ART / manifest[key]
    if not path.is_file() or path.stat().st_size < 1024:
        raise ValueError(f"{manifest[key]} is missing or implausibly small")
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != manifest[key + "Sha256"]:
        raise ValueError(f"{manifest[key]} SHA-256 {digest} != manifest {manifest[key + 'Sha256']}")
    if expected_override and digest != expected_override.strip().lower():
        raise ValueError(f"{manifest[key]} SHA-256 {digest} != expected {expected_override}")
with zipfile.ZipFile(ART / manifest["fittedEstimator"]) as z:
    names = z.namelist()
    for n in names:
        p = PurePosixPath(n.replace("\\", "/"))
        if p.is_absolute() or ".." in p.parts:
            raise ValueError(f"unsafe member in fitted archive: {n!r}")
    if "init_params.json" not in names:
        raise ValueError("fitted archive lacks init_params.json")
    init_params = json.loads(z.read("init_params.json"))
FEATURES = list(manifest["featureColumns"])
print("Validated artifact:", {k: manifest.get(k) for k in ("schemaVersion", "taskType", "targetColumn", "portableLoader")})
print({"featureCount": len(FEATURES), "features": FEATURES, "classes": manifest.get("classes"), "fittedSha256": manifest["fittedEstimatorSha256"][:16], "ckptSha256": manifest["foundationCheckpointSha256"][:16], "recordedModelPath": init_params.get("model_path")})

## 2. Reconstruct the serving estimator

`load_dimer_tabpfn_artifact()` rewrites the fitted archive's recorded `model_path` — in a temporary copy, never the original — to point at the companion `model.ckpt` beside it, then calls TabPFN's `load_fitted_tabpfn_model()`. No network download is attempted for the foundation weights: they come from the uploaded checkpoint. The in-context training rows and any fine-tuned weights travel inside the artifact; nothing is refit here.

In [ ]:
model = load_dimer_tabpfn_artifact(ART, device="cuda" if torch.cuda.is_available() else "cpu")
print("reconstructed:", type(model).__name__, "| n_estimators:", getattr(model, "n_estimators", None), "| device:", getattr(model, "device", None))
if IS_CLASSIFIER:
    classes = [str(c) for c in getattr(model, "classes_", [])]
    if classes and classes != [str(c) for c in manifest.get("classes", [])]:
        raise RuntimeError("reconstructed classes disagree with the manifest")
    print("classes:", classes)

## 3. Upload, validate, and score genuinely new input

Upload one CSV containing exactly the artifact's feature columns, with no target column or pre-existing `prediction`/`proba_*` columns. Duplicate or missing columns fail before prediction; extra columns are rejected rather than silently dropped. Numeric columns are checked for non-numeric and infinite values. Output columns: `prediction` is the raw point estimate; no interval column exists because the pipeline provides none.

In [ ]:
import csv, io
new_upload = files.upload()
if len(new_upload) != 1:
    raise ValueError("Upload exactly one CSV.")
input_name, raw = next(iter(new_upload.items()))
header = next(csv.reader(io.StringIO(raw.decode("utf-8-sig"))), [])
duplicates = sorted({x for x in header if header.count(x) > 1})
if duplicates:
    raise ValueError(f"Duplicate CSV columns: {duplicates}")
new_data = pd.read_csv(io.BytesIO(raw))
reserved = [manifest["targetColumn"], "prediction", *[f"proba_{c}" for c in manifest.get("classes", [])]]
present = [c for c in reserved if c in new_data.columns]
if present:
    raise ValueError(f"Remove target/prediction/proba columns before inference: {present}")
missing = [c for c in FEATURES if c not in new_data.columns]
extra = [c for c in new_data.columns if c not in FEATURES]
if missing or extra:
    raise ValueError(f"Feature schema mismatch; missing={missing}, extra={extra}")
new_data = new_data.loc[:, FEATURES].copy()
for column in new_data.select_dtypes(include=np.number).columns:
    values = new_data[column].dropna().to_numpy(dtype=float)
    if values.size and not np.isfinite(values).all():
        raise ValueError(f"Numeric feature {column!r} contains infinite values.")
print("Input rows/features:", new_data.shape, "| missing:", new_data.isna().sum()[lambda s: s > 0].to_dict() or "none")
pred = np.asarray(model.predict(new_data))
results = pd.DataFrame({"row_id": new_data.index.to_numpy(), "prediction": pred})
if IS_CLASSIFIER:
    proba = np.asarray(model.predict_proba(new_data))
    for i, label in enumerate(manifest["classes"]):
        results[f"proba_{label}"] = proba[:, i]
results.head()

## 4. Export predictions and provenance

`row_id` maps each prediction to its input row. Provenance records the externally supplied artifact identity (manifest fields and digests), the runtime, the reconstruction device, and the input shape; it contains no credentials.

In [ ]:
OUT = Path("/content/tabpfn-artifact-inference-output")
OUT.mkdir(parents=True, exist_ok=True)
results.to_csv(OUT / "tabpfn_regression_predictions.csv", index=False)
provenance = {
    "artifact": {k: manifest.get(k) for k in ("schemaVersion", "taskType", "targetColumn", "classes", "fittedEstimator", "fittedEstimatorSha256", "foundationCheckpoint", "foundationCheckpointSha256", "portableLoader")},
    "reconstruction": {"loader": "serving/load_artifact.py:load_dimer_tabpfn_artifact", "device": "cuda" if torch.cuda.is_available() else "cpu", "networkFallbackForWeights": False, "refit": False},
    "runtime": {"python": platform.python_version(), "torch": md.version("torch"), "tabpfn": md.version("tabpfn"), "pandas": md.version("pandas")},
    "input": {"filename": input_name, "rows": int(len(new_data)), "features": FEATURES},
}
(OUT / "tabpfn_regression_inference_provenance.json").write_text(json.dumps(provenance, indent=2, default=str) + "\n")
print("Wrote prediction CSV and provenance JSON to", OUT)

## Interpretation and troubleshooting

A successful run proves an independently supplied artifact bundle is internally consistent with its manifest, that the serving loader reconstructs the estimator from the bundle alone without refitting or downloading weights, and that schema-compatible new rows can be scored and exported. It does **not** authenticate the producer, make untrusted serialized estimator state safe to load, or establish predictive quality, uncertainty, fairness, or production fitness. Never bypass a failed digest, schema, or archive-safety check; obtain a correct artifact from a trusted producer. If reconstruction fails with a tabpfn/torch import or version error, install the exact `tutorials/requirements-colab.txt` lock set the producer used.